In [ ]:
# ============================================================================
# CELL 1: SETUP & CONNECTION
# ============================================================================
# Initialize broker, beacon, and all MQTT device clients.
# Run this first before any device tests.
# ============================================================================

import time
import sys
from pathlib import Path

# Add parent directory (monorepo root) to path
notebook_dir = Path.cwd()
monorepo_root = notebook_dir.parent if notebook_dir.name == "notebooks" else notebook_dir
sys.path.insert(0, str(monorepo_root))

from src.adapters.iot_mqtt import PumpMQTT, UltraMQTT, HeatMQTT, ReactorMQTT, start_broker_if_needed, stop_broker, ControllerBeacon, _best_effort_all_off

# ---- Start Broker ----
proc = start_broker_if_needed()  # Only if you don't run the Windows service

# ---- Broker Configuration ----
broker = "localhost"  # Broker running on this machine
port = 1883
username = "pyctl-controller"
password = "controller"

# ---- Controller Beacon (heartbeat supervisor) ----
beacon = ControllerBeacon(
    broker=broker, port=port, 
    username=username, password=password, 
    client_id="pyctl-controller",
    status_topic="pyctl/status", 
    heartbeat_topic="pyctl/heartbeat", 
    heartbeat_interval=5.0, 
    keepalive=30
)
beacon.start()
print("[BEACON] Controller heartbeat started")

# ---- Device Initialization ----
pumps = PumpMQTT(
    broker=broker, username=username, password=password,
    base_topic="pumps/01", client_id="pyctl-pumps"
)
ultra = UltraMQTT(
    broker=broker, username=username, password=password,
    base_topic="ultra/01", client_id="pyctl-ultra"
)
heat = HeatMQTT(
    broker=broker, username=username, password=password,
    base_topic="heat/01", client_id="pyctl-heat"
)
reactor = ReactorMQTT(
    broker=broker, username=username, password=password,
    base_topic="reactor/01", client_id="pyctl-reactor"
)

# ---- Connect All Devices ----
pumps.ensure_connected()
ultra.ensure_connected()
heat.ensure_connected()
reactor.ensure_connected()
time.sleep(1)  # Wait for connections to settle

print("\n" + "="*60)
print("✓ Connected to broker at", broker)
print("✓ All devices initialized and ready")
print("="*60)

In [ ]:
# ============================================================================
# CELL 2: DEVICE STATUS CHECK
# ============================================================================
# Queries all devices for current status and heartbeat.
# Use this to verify all devices are online and responding.
# ============================================================================

print("\n[STATUS] Polling all devices...\n")

print("> Pumps (3-channel relay):")
pumps.status(seconds=2.0)
time.sleep(1)

print("> Ultrasonic (2-channel relay):")
ultra.status(seconds=2.0)
time.sleep(1)

print("> Heater (thermistor + PID control):")
heat.status(seconds=2.0)
time.sleep(1)

print("> Reactor (forward/reverse actuator):")
reactor.status(seconds=2.0)

print("\n[STATUS] Device check complete")

In [ ]:
# ============================================================================
# CELL 3: PUMPS TEST (Peristaltic Pumps)
# ============================================================================
# Tests the 3-channel pump relay module.
# Capabilities:
# - pumps.on(ch) - Continuous ON (max 60s lease)
# - pumps.off(ch) - Manual OFF
# - pumps.on(ch, ms) - Timed activation with auto-off
# ============================================================================

print("\n[PUMPS]" + "="*56)
print("Testing pump relay module (3 channels)\n")

# TIMED TEST (default)
pumps.on(3, 5000)           # Pump 3 ON for 5s, then auto-OFF
print("→ Pump 3 activated for 5s (auto-off)")

# MANUAL CONTROL (uncomment to test)
# pumps.on(3)                 # Pump 3 ON (continuous, max 60s lease)
# time.sleep(1)
# pumps.off(3)                # Pump 3 OFF (manual stop)
# time.sleep(1)

# pumps.on(2)                 # Pump 2 ON
# time.sleep(1)
# pumps.off(2)                # Pump 2 OFF
# time.sleep(1)

# pumps.on(1)                 # Pump 1 ON
# time.sleep(1)
# pumps.off(1)                # Pump 1 OFF

print("[PUMPS] Test complete\n")

In [ ]:
# ============================================================================
# CELL 4: ULTRASONIC TEST (Sonication)
# ============================================================================
# Tests the 2-channel ultrasonic relay module.
# Can be used for: sonication, mixing, agitation.
# Capabilities:
# - ultra.on(ch) - Continuous ON (max 60s lease)
# - ultra.off(ch) - Manual OFF
# - ultra.on_for(ch, ms) - Timed with auto-off
# ============================================================================

print("\n[ULTRASONIC]" + "="*50)
print("Testing ultrasonic relay module (2 channels)\n")

# TIMED TEST (default)
ultra.on(2)                 # Ultrasonic ch2 ON (30s default)
print("→ Ultrasonic ch2 activated")
time.sleep(20)              # Let it run for 20s
ultra.off(2)                # Manual OFF
print("→ Ultrasonic ch2 deactivated")

# ALTERNATIVE: Other channel (uncomment to test)
# ultra.on(1)                # Ultrasonic ch1 ON
# time.sleep(20)
# ultra.off(1)               # Ultrasonic ch1 OFF

print("[ULTRASONIC] Test complete\n")

In [ ]:
# Heater + thermistor demo
heat.set_target(1, 62.0)         # set target to 42C (ESP retains on set/1)
heat.pid_on(1)                   # enable PID loop on ESP
# actively request a reading now:
try:
    for i in range(500):
        t = heat.get_base_temp(1, timeout_s=5.0)
        print("Temp(ch1) =", t)
        time.sleep(1)
except TimeoutError as e:
    print("Temp read timeout:", e)

heat.pid_off(1)                  # stop PID
heat.set_pwm(1, 0)               # ensure PWM is off
heat.off(1)                      # ensure relay is off

In [ ]:
heat.on(0)

In [ ]:
set_time = 8000
pumps.on(1, set_time)           # pump 1 ON for 2 s (auto-off)
pumps.on(2, set_time)           # pump 1 ON for 2 s (auto-off)
pumps.on(3, set_time)           # pump 1 ON for 2 s (auto-off)

ultra.on(2, set_time)       # ultrasonic ch2 ON for 30 s
ultra.on(1, set_time)       # ultrasonic ch2 ON for 30 s

heat.set_target(1, 42.0)         # set target to 42C (ESP retains on set/1)
# heat.pid_on(1)                   # enable PID loop on ESP
# time.sleep(set_time/1000)
# heat.pid_off(1)                  # stop PID
# heat.set_pwm(1, 0)               # ensure PWM is off
# heat.off(1)                      # ensure relay is off

In [20]:
reactor.forward(5000)
time.sleep(6)

reactor.reverse(5000)
time.sleep(6)

[pyctl-reactor] Published 'FORWARD:5000' to reactor/01/cmd/1
[pyctl-reactor] Published 'REVERSE:5000' to reactor/01/cmd/1


In [ ]:
_best_effort_all_off(pumps, ultra, heat, reactor)

pumps.disconnect()
ultra.disconnect()
heat.disconnect()
reactor.disconnect()

beacon.stop()
stop_broker(proc)